# Exercise for Unit 4.1: Naïve Bayes
### Part 1 of the Task: Text Classification for SPAM and HAM

In [25]:
#importing needed libraries and preprocesing the data
import math
import re

#dataset from the unit exercise document
documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

def tokenize(text):
    """Converts text to lowercase and extracts alphanumeric words."""
    return re.findall(r'\b\w+\b', text.lower())

In [26]:
#Number 1A: Generating a Bag of Words (for word frequency)
def generate_bow(docs):
    vocab = set()
    word_counts = {"SPAM": {}, "HAM": {}}
    class_total_words = {"SPAM": 0, "HAM": 0} 
    
    for text, label in docs:
        tokens = tokenize(text)
        for token in tokens:
            vocab.add(token)
            #increment the count for this word in the specific class
            word_counts[label][token] = word_counts[label].get(token, 0) + 1
            #increment the total word count for the class
            class_total_words[label] += 1
            
    return vocab, word_counts, class_total_words

vocab, word_counts, class_total_words = generate_bow(documents)
print(f"Vocabulary size: {len(vocab)} unique words")

Vocabulary size: 45 unique words


**Explanation:** The model has identified 45 distinct words across all training documents after lowercasing and removing punctuation. This vocabulary acts as the feature set for our model, and every calculation we perform from here on is based on these 45 specific features.

In [27]:
#Number 1B: Calculate the prior for the class HAM and SPAM
def calculate_priors(docs):
    total_docs = len(docs)
    spam_docs = sum(1 for _, label in docs if label == "SPAM")
    ham_docs = total_docs - spam_docs
    
    return {"SPAM": spam_docs / total_docs, "HAM": ham_docs / total_docs}

priors = calculate_priors(documents)
print(f"Priors: {priors}")

Priors: {'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454}


**Explanation:** Here shows the prior, which is our baseline probability. This is based purely on the training data (5 SPAM messages and 6 HAM messages), there is a **45.45%** chance any given message is SPAM and a **54.55%** chance it is HAM before we even read the content.

In [28]:
#function to compute likelihood with Laplace smoothing (for number 1C task)
def calculate_likelihood(word, label, vocab, word_counts, class_total_words):
    
    # Count of the word in the given class
    word_count = word_counts[label].get(word, 0)
    
    # Total number of words in that class
    total_words = class_total_words[label]
    
    # Vocabulary size
    vocab_size = len(vocab)
    
    # Laplace smoothing formula
    prob = (word_count + 1) / (total_words + vocab_size)
    
    return prob

In [29]:
#Number 1C: Calculate the likelihood of the tokens in the vocabulary with respect to the class
def get_all_likelihoods(vocab, word_counts, class_total_words):
    likelihoods = {"SPAM": {}, "HAM": {}}

    for label in ["SPAM", "HAM"]:
        for word in vocab:
            # We call the function we already built to do the math
            prob = calculate_likelihood(word, label, vocab, word_counts, class_total_words)
            likelihoods[label][word] = prob

    return likelihoods

# Generate the full likelihood table
all_likelihoods = get_all_likelihoods(vocab, word_counts, class_total_words)

# Displaying small samples of the likelihoods for both classes to verify
print("Sample Likelihoods (Top 5 words in SPAM):")
spam_sorted = sorted(all_likelihoods["SPAM"].items(), key=lambda x: x[1], reverse=True)[:5]
for word, prob in spam_sorted:
    print(f"P({word} | SPAM) = {prob:.4f}")

print("\nSample Likelihoods (Top 5 words in HAM):")
ham_sorted = sorted(all_likelihoods["HAM"].items(), key=lambda x: x[1], reverse=True)[:5]
for word, prob in ham_sorted:
    print(f"P({word} | HAM) = {prob:.4f}")

Sample Likelihoods (Top 5 words in SPAM):
P(free | SPAM) = 0.0448
P(for | SPAM) = 0.0448
P(off | SPAM) = 0.0299
P(here | SPAM) = 0.0299
P(50 | SPAM) = 0.0299

Sample Likelihoods (Top 5 words in HAM):
P(the | HAM) = 0.0506
P(tomorrow | HAM) = 0.0380
P(you | HAM) = 0.0380
P(at | HAM) = 0.0380
P(office | HAM) = 0.0380


**Explanation:** This cell represents the likelihood. Likelihood represents the probability of a word appearing given a specific class. 
* In the **SPAM** results, words like "free" and "for" have higher probabilities, showing they are strong indicators of spam. 
* In the **HAM** results, functional words like "the", "tomorrow", and "office" dominate. 

**Extra Note on Laplace Smoothing:** Even though "office" might never have appeared in a SPAM message, its probability is not zero (thanks to our `+1` smoothing), preventing the model from failing when it encounters "office" in a potential spam email.

In [30]:
#Number 1D: Determine the Class of the Test Sentences
def predict_naive_bayes(text, vocab, word_counts, class_total_words, priors):
    tokens = tokenize(text)
    scores = {}
    
    for label in ["SPAM", "HAM"]:
        # Start with the log of the prior probability
        score = math.log(priors[label])
        for token in tokens:
            # Only consider words that exist in our training vocabulary
            if token in vocab:
                prob = calculate_likelihood(token, label, vocab, word_counts, class_total_words)
                score += math.log(prob)
        scores[label] = score
        
    # Return the class with the highest score
    return "SPAM" if scores["SPAM"] > scores["HAM"] else "HAM"

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
    "Congratulations, you've won a prize!",
    "Don't miss out on this limited time deal.",
    "Can we reschedule our meeting to Thursday?",
    "Reminder: project deadline is next Monday.",
    "Prepare the minutes of the meeting by 5 PM.",
    "Exclusive: claim your free voucher now.",
    "Are you available for a call tomorrow?",
    "Save big on your subscription today.",
    "Unlimited coupons, claim them here.",
    "Up to 80% off on all products, shop now!",
    "Lunch at noon?"
]

print("--- Naïve Bayes Predictions ---")
for sentence in test_sentences:
    prediction = predict_naive_bayes(sentence, vocab, word_counts, class_total_words, priors)
    print(f"'{sentence}' -> {prediction}")

--- Naïve Bayes Predictions ---
'Limited offer, click here!' -> SPAM
'Meeting at 2 PM with the manager.' -> HAM
'Congratulations, you've won a prize!' -> HAM
'Don't miss out on this limited time deal.' -> SPAM
'Can we reschedule our meeting to Thursday?' -> HAM
'Reminder: project deadline is next Monday.' -> HAM
'Prepare the minutes of the meeting by 5 PM.' -> HAM
'Exclusive: claim your free voucher now.' -> SPAM
'Are you available for a call tomorrow?' -> HAM
'Save big on your subscription today.' -> SPAM
'Unlimited coupons, claim them here.' -> SPAM
'Up to 80% off on all products, shop now!' -> SPAM
'Lunch at noon?' -> HAM


### Analysis of our Manual Prediction Testing
**Explanation:** The manual model successfully distinguishes between promotional language (talks about sales) and professional/personal language (like some sort of organization or company). 
* For example in **"Limited offer..."** was correctly classified as **SPAM** because the cumulative log-probabilities of those specific words were higher for the SPAM class.
* Also for **"Meeting at 2 PM..."** was correctly classified as **HAM** because the model associated "meeting" and "at" more strongly with the HAM category.

## Part 2: Implementation using Scikit-Learn

In [31]:
#Train and test using Scikit-Learn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

#separate text and labels
texts = [doc[0] for doc in documents]
labels = [doc[1] for doc in documents]

#initialize and fit the Bag of Words vectorizer
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(texts)

#initialize and train the Multinomial Naïve Bayes classifier
clf = MultinomialNB()
clf.fit(X_train, labels)


#Number 2A: Determine the class of the test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
    "Congratulations, you've won a prize!",
    "Don't miss out on this limited time deal.",
    "Can we reschedule our meeting to Thursday?",
    "Reminder: project deadline is next Monday.",
    "Prepare the minutes of the meeting by 5 PM.",
    "Exclusive: claim your free voucher now.",
    "Are you available for a call tomorrow?",
    "Save big on your subscription today.",
    "Unlimited coupons, claim them here.",
    "Up to 80% off on all products, shop now!",
    "Lunch at noon?"
]

# Vectorize the test sentences using the SAME vectorizer
X_test = vectorizer.transform(test_sentences)

# Predict
predictions = clf.predict(X_test)

print("--- Scikit-Learn Predictions ---")
for sentence, pred in zip(test_sentences, predictions):
    print(f"'{sentence}' -> {pred}")

--- Scikit-Learn Predictions ---
'Limited offer, click here!' -> SPAM
'Meeting at 2 PM with the manager.' -> HAM
'Congratulations, you've won a prize!' -> HAM
'Don't miss out on this limited time deal.' -> SPAM
'Can we reschedule our meeting to Thursday?' -> HAM
'Reminder: project deadline is next Monday.' -> HAM
'Prepare the minutes of the meeting by 5 PM.' -> HAM
'Exclusive: claim your free voucher now.' -> SPAM
'Are you available for a call tomorrow?' -> HAM
'Save big on your subscription today.' -> SPAM
'Unlimited coupons, claim them here.' -> SPAM
'Up to 80% off on all products, shop now!' -> SPAM
'Lunch at noon?' -> HAM


### Analysis of Part 2: Scikit-Learn Implementation
**Explanation:** The Scikit-Learn `MultinomialNB` results match our manual implementation exactly. This confirms that our manual mathematical logic (using Laplace smoothing and Log-probabilities) is aligned with industry-standard library implementations. Using `CountVectorizer` and `MultinomialNB` allows us to scale this logic to thousands of documents with just a few lines of code.